In [1]:
import sys
path = "../../.."
if path not in sys.path:
    sys.path.insert(0, path)

from data_retrieval import lipade_groundtruth
from PIL import Image
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from data_retrieval.tools.data_loader import getDataLoader
from torch.utils.data import DataLoader
import torch.optim as optim
from degradations.methods import transforms_blue, transforms_faded, transforms_rectangular_frame, transforms_atkinson_dithering, transforms_bayer_halftoning, transforms_floyd_steinberg_halftoning, transforms_drawing, transforms_erased_element, transforms_paint, transforms_non_rectangular_frame, transforms_patchwork, transforms_photo_montage, transforms_picture_overlay, transforms_text_overlay, transforms_dirty_rollers, transforms_add_gaussian_noise, transforms_add_salt_and_pepper_noise, transforms_bleedthrough, transforms_contrast, transforms_crumpled_paper, transforms_folded_paper, transforms_ink_bleed,  transforms_book, transforms_stains, transforms_scribbles, transforms_torn_paper, transforms_text_around


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


corpus = "lipade_groundtruth"
rawPath = "../results/raw/" + corpus + "/"


x,_,y = lipade_groundtruth.getDataset(mode = 'unique', uniform=True)

for i in range(len(x)):
    try:
        x[i] = Image.open(x[i]).convert('RGB')
    except:
        print("Error loading image:", x[i])

images = np.array(x)


class transforms_SepiaFilter(nn.Module):
    def __init__(self):
        super(transforms_SepiaFilter, self).__init__()

    def __call__(self, batch):
        sepia_filter = torch.tensor([[0.393, 0.769, 0.189],
                                     [0.349, 0.686, 0.168],
                                     [0.272, 0.534, 0.131]], device=batch.device)
        one_image = False
        if len(batch.shape) == 3: # si on ne passe qu'une image au lieu d'un batch
            batch = batch.unsqueeze(0)
            one_image = True

        batch = torch.einsum('ijkl,mj->imkl', batch, sepia_filter)

        if one_image:
            batch = batch.squeeze(0)
        return batch.clamp(0, 1)

transform_1by1 = transforms.Compose([
        transforms.RandomChoice([
            transforms.RandomResizedCrop(size=images.shape[2], scale=(1/2, 1), ratio=(1, 1)),
            transforms_floyd_steinberg_halftoning(),
            transforms_atkinson_dithering(),
            transforms_bayer_halftoning(),
            transforms_picture_overlay(),
            transforms_text_overlay(),
            transforms_text_around(),
            transforms_rectangular_frame(),
            transforms_non_rectangular_frame(),
            transforms_torn_paper(),
            transforms_erased_element(),
            transforms_add_gaussian_noise(),
            transforms_add_salt_and_pepper_noise(),
            transforms_faded(),
            transforms_dirty_rollers(),
            transforms_stains(),
            #transforms_ink_bleed(),
            transforms_bleedthrough(),
            transforms_contrast(),
            transforms_blue(),
            transforms_drawing(),
            transforms_patchwork(),
            transforms_paint(),
            #transforms_crumpled_paper(),
            #transforms_folded_paper(), #
            transforms.RandomHorizontalFlip(p=1),
            transforms.RandomVerticalFlip(p=1),
            transforms_SepiaFilter()]),
        transforms.ColorJitter(brightness=0.8, contrast=0.8, saturation=0.8, hue=0.2),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=5)], p=0.5),
])



In [2]:
from degradations.methods import transforms_blue, transforms_faded, transforms_rectangular_frame, transforms_atkinson_dithering, transforms_bayer_halftoning, transforms_floyd_steinberg_halftoning, transforms_drawing, transforms_erased_element, transforms_paint, transforms_non_rectangular_frame, transforms_patchwork, transforms_photo_montage, transforms_picture_overlay, transforms_text_overlay, transforms_dirty_rollers, transforms_add_gaussian_noise, transforms_add_salt_and_pepper_noise, transforms_bleedthrough, transforms_contrast, transforms_crumpled_paper, transforms_folded_paper, transforms_ink_bleed,  transforms_book, transforms_stains, transforms_scribbles, transforms_torn_paper, transforms_text_around

transform_1by1 = transforms.Compose([
        transforms.RandomChoice([
            transforms.RandomResizedCrop(size=images.shape[2], scale=(1/2, 1), ratio=(1, 1)),
            transforms_floyd_steinberg_halftoning(),
            transforms_atkinson_dithering(),
            transforms_bayer_halftoning(),
            transforms_picture_overlay(),
            transforms_text_overlay(),
            transforms_text_around(),
            transforms_rectangular_frame(),
            transforms_non_rectangular_frame(),
            transforms_torn_paper(),
            transforms_erased_element(),
            transforms_add_gaussian_noise(),
            transforms_add_salt_and_pepper_noise(),
            transforms_faded(),
            transforms_dirty_rollers(),
            transforms_stains(),
            #transforms_ink_bleed(),
            transforms_bleedthrough(),
            transforms_contrast(),
            transforms_blue(),
            #transforms_drawing(),
            #transforms_patchwork(),
            transforms_paint(),
            #transforms_crumpled_paper(),
            #transforms_folded_paper(), #
            transforms.RandomHorizontalFlip(p=1),
            transforms.RandomVerticalFlip(p=1),
            transforms_SepiaFilter()]),
        transforms.ColorJitter(brightness=0.8, contrast=0.8, saturation=0.8, hue=0.2),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=5)], p=0.5),
])

In [3]:
import torchvision
import os

for i, image in enumerate(tqdm(images)):
    image = transforms.ToTensor()(image)

    if i <2461:
        continue
    else:
        if not os.path.exists(f"corpus_degrad/{i}"):
            os.makedirs(f"corpus_degrad/{i}")
        for j in range(10):
            #degraded_image = torchvision.transforms.ToPILImage()(image)
            degraded_image = transform_1by1(image).numpy() * 255
            degraded_image = degraded_image.astype(np.uint8).transpose(1, 2, 0)
            im = Image.fromarray(degraded_image)
            im.save(f"corpus_degrad/{i}/{j}.jpg")


100%|██████████| 5855/5855 [1:27:03<00:00,  1.12it/s] 
